# MO-IT162 Milestone 2: ML Solution Data Visualization

**Project:** FinMark Corporation  
**Section:** A3101 | **Team:** Team 3  
**Course:** Math for Machine Learning: Linear Algebra

This submission extends Milestone 1 and Week 7 visualization exploration. It uses only the finalized Week 3 cleaned datasets. The focus is on market activity, category performance, transaction value, customer participation, social sentiment, and cross-channel behavior.

> Run all cells from top to bottom in Colab/Jupyter. The narrative distinguishes dataset evidence from interpretation; correlations are associations and do not establish causation.

## Mentor feedback and revision record

The Milestone 2 draft and mentor feedback were not present among the supplied project files. This version therefore applies the supplied proposal outline and Week 7 findings as the revision basis. Add the mentor’s comments and map each comment to a change before submission if those materials are available.

## 1. Project context and business problem

FinMark has customer demographic, transaction, and social interaction records but needs a clear view of market activity and customer behavior to support data-informed decisions. Milestone 1 established data quality, distributions, temporal coverage, and initial relationships. Week 7 compared candidate visualizations. This milestone narrows that exploration to decision-relevant charts and validates the associated interpretations.

**Analytical goal:** describe changes in transaction activity and sentiment, compare product activity and value, and test whether customer participation and social engagement correspond with purchasing behavior. This is descriptive analysis, not a causal or predictive model.

## 2. Scope and analytical boundaries

Included: monthly transaction activity, category counts and value, transaction amount distribution, gender-category participation, monthly sentiment composition, and customer-level engagement/purchasing relationships. Excluded from the main story: signup tenure (date inconsistency), standalone platform/payment frequencies, full correlation heatmaps, and redundant demographic charts. Identifiers are join keys only. Event records are aggregated before customer-level joins. Transaction-social comparisons use their shared observation period.

## 3. Data loading and verification

The data directory is relative to this notebook. If the notebook is uploaded separately, update `DATA_DIR` to the cleaned CSV folder.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

sns.set_theme(style="whitegrid", palette="colorblind")
DATA_DIR = Path("Week3_Preprocessing_Submission")
if not DATA_DIR.exists():
    DATA_DIR = Path("A3101-ml-linear-algebra-project-Team-3-main/Week3_Preprocessing_Submission")

demographics = pd.read_csv(DATA_DIR / "customer_demographics_cleaned.csv", parse_dates=["signup_date"])
transactions = pd.read_csv(DATA_DIR / "customer_transactions_cleaned.csv", parse_dates=["transaction_date"])
social = pd.read_csv(DATA_DIR / "social_media_interactions_cleaned.csv", parse_dates=["interaction_date"])

expected = {
    "demographics": {"customer_id", "age", "gender", "income_level", "signup_date"},
    "transactions": {"customer_id", "transaction_id", "transaction_date", "amount", "product_category"},
    "social": {"customer_id", "interaction_id", "interaction_date", "sentiment"},
}
frames = {"demographics": demographics, "transactions": transactions, "social": social}
for name, df in frames.items():
    missing_cols = expected[name] - set(df.columns)
    print(f"{name}: rows={len(df):,}, columns={len(df.columns)}, null cells={df.isna().sum().sum():,}, exact duplicate rows={df.duplicated().sum():,}")
    assert not missing_cols, f"{name} missing columns: {missing_cols}"
print("Transaction IDs unique:", transactions.transaction_id.is_unique)
print("Interaction IDs unique:", social.interaction_id.is_unique)
print("Date spans:", transactions.transaction_date.min().date(), "to", transactions.transaction_date.max().date(), "|", social.interaction_date.min().date(), "to", social.interaction_date.max().date())


## 4. Analytical preparation

Transaction and social records are kept at event level for their own monthly and category summaries. For cross-channel analysis, each source is first restricted to the shared date interval, aggregated to one row per customer, then inner-joined. Participation uses the demographics-and-transactions matched population; a customer counts as a category participant if at least one purchase is recorded.


In [ ]:
common_start = max(transactions.transaction_date.min(), social.interaction_date.min())
common_end = min(transactions.transaction_date.max(), social.interaction_date.max())
tx_common = transactions.loc[transactions.transaction_date.between(common_start, common_end)].copy()
social_common = social.loc[social.interaction_date.between(common_start, common_end)].copy()

tx_customer = tx_common.groupby("customer_id").agg(
    transaction_count=("transaction_id", "count"), total_spend=("amount", "sum"),
    average_transaction_amount=("amount", "mean")
).reset_index()
social_customer = social_common.groupby("customer_id").agg(
    interaction_count=("interaction_id", "count")
).reset_index()
matched = tx_customer.merge(social_customer, on="customer_id", how="inner", validate="one_to_one")
print(f"Shared observation window: {common_start.date()} to {common_end.date()}")
print(f"Transactions in window: {len(tx_common):,}; interactions in window: {len(social_common):,}")
print(f"Matched customers in aligned comparison: {len(matched):,}; duplicate customer rows: {matched.customer_id.duplicated().sum()}")

# One row per demographics customer with recorded transactions; event-level rows are not used as independent customers.
customer_tx = demographics[["customer_id", "gender"]].merge(
    transactions[["customer_id", "product_category"]].drop_duplicates(),
    on="customer_id", how="inner", validate="one_to_many"
)
category_population = demographics[["customer_id", "gender"]].merge(
    transactions[["customer_id"]].drop_duplicates(), on="customer_id", how="inner", validate="one_to_one"
)
print(f"Gender-category participation population: {len(category_population):,} matched customers")


## 5. Market activity over time

**Question:** How did transaction count and recorded value change by month? The paired line charts use the same month sequence and distinguish event count from total value. Peaks and troughs are descriptive; 24 months are not sufficient to establish a reliable recurring seasonal pattern.


In [ ]:
monthly_tx = (tx_common.assign(month=tx_common.transaction_date.dt.to_period("M").dt.to_timestamp())
              .groupby("month").agg(transaction_count=("transaction_id", "count"), total_value=("amount", "sum")).reset_index())
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sns.lineplot(data=monthly_tx, x="month", y="transaction_count", marker="o", ax=axes[0], color="#286090")
axes[0].set(title="Monthly transaction activity", xlabel="", ylabel="Transactions")
sns.lineplot(data=monthly_tx, x="month", y="total_value", marker="o", ax=axes[1], color="#b45309")
axes[1].set(title="Monthly recorded transaction value", xlabel="Month", ylabel="Total amount")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("Monthly count range:", monthly_tx.transaction_count.min(), "to", monthly_tx.transaction_count.max())
print("Peak count month:", monthly_tx.loc[monthly_tx.transaction_count.idxmax(), "month"].strftime("%Y-%m"))
print("Lowest count month:", monthly_tx.loc[monthly_tx.transaction_count.idxmin(), "month"].strftime("%Y-%m"))
display(monthly_tx.head())


**Interpretation:** Read transaction count and total value together: a change in total value can reflect a different number of transactions, different ticket sizes, or both. Week 7 found monthly counts ranged from 82 to 121 across the full July 2022–June 2024 series, with May 2024 highest (121) and February 2023 lowest (82). These are observed fluctuations, not evidence of seasonality or their causes.

**FinMark implication:** Use the monthly series as a baseline for monitoring. Investigate unusual months alongside campaign, inventory, and operating records before attributing a cause.

## 6. Product-category performance

**Question:** Which categories account for the largest transaction volume and recorded value? Counts show activity; summed amounts show recorded monetary contribution. These measures are not customer counts and do not by themselves measure profitability.


In [ ]:
category_summary = (transactions.groupby("product_category").agg(
    transaction_count=("transaction_id", "count"), total_value=("amount", "sum"),
    median_amount=("amount", "median"), average_amount=("amount", "mean")
).sort_values("transaction_count", ascending=True))
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
category_summary.transaction_count.plot.barh(ax=axes[0], color="#286090")
axes[0].set(title="Transaction count by product category", xlabel="Transactions", ylabel="")
category_summary.total_value.plot.barh(ax=axes[1], color="#b45309")
axes[1].set(title="Total recorded transaction value by category", xlabel="Total amount", ylabel="")
plt.tight_layout(); plt.show()
display(category_summary.sort_values("transaction_count", ascending=False).round(2))
print("Count share range (%):", round(100*category_summary.transaction_count.min()/category_summary.transaction_count.sum(),1), "to", round(100*category_summary.transaction_count.max()/category_summary.transaction_count.sum(),1))


**Interpretation:** Week 7 found category transaction shares were close (18.93%–20.87%), so no category should be called dominant based on frequency alone. The paired count/value chart tests whether value ranking differs from activity ranking; consult the displayed summary before describing that gap.

**FinMark implication:** Compare activity and value when planning category-level review. Revenue is not profit, and category totals can reflect both volume and ticket size.

## 7. Transaction-amount distribution and unusual values

**Question:** What is the spread of individual recorded transaction amounts? The histogram is paired with robust and conventional summaries. The Week 3 cleaning rule converted explicit `Free` amounts to zero and removed unsupported negative amounts.


In [ ]:
amount = transactions.amount.dropna()
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(amount, bins=20, ax=ax, color="#286090")
ax.axvline(amount.median(), color="#b45309", linestyle="--", label=f"Median {amount.median():.2f}")
ax.axvline(amount.mean(), color="#7c3aed", linestyle=":", label=f"Mean {amount.mean():.2f}")
ax.set(title="Distribution of individual transaction amounts", xlabel="Recorded amount", ylabel="Transactions")
ax.legend(); plt.tight_layout(); plt.show()
q1, q3 = amount.quantile([.25,.75]); iqr=q3-q1
amount_stats = pd.Series({"n": amount.size, "mean": amount.mean(), "median": amount.median(), "std": amount.std(), "min": amount.min(), "Q1": q1, "Q3": q3, "IQR": iqr, "max": amount.max(), "zero_count": int((amount==0).sum()), "IQR_flag_count": int(((amount < q1-1.5*iqr)|(amount > q3+1.5*iqr)).sum())})
display(amount_stats.to_frame("Value").round(2))


**Interpretation:** Milestone 1 reported mean 495.25, median 502.13, SD 295.71, range 0–999.86, and no observations beyond the 1.5×IQR bounds. The zero mode (54 observations in that analysis) includes the documented `Free` normalization, so zero should not be presented as a typical paid purchase. Statistical outlier rules flag unusual values for review; they do not prove data errors.

**FinMark implication:** The broad spread supports reporting typical values with medians and distributions, not only averages. Retain zero semantics in any later feature engineering.

## 8. Gender and product-category participation

**Question:** Is recorded gender associated with purchasing at least once from a product category? Unit: one matched customer per category test. The category-specific participation percentages and chi-square tests are accompanied by Bonferroni-adjusted p-values and Cramér’s V so small effects are not overstated. Gender is diagnostic, not a targeting prescription.


In [ ]:
category_rows=[]
for cat in sorted(transactions.product_category.dropna().unique()):
    buyers=set(transactions.loc[transactions.product_category.eq(cat), "customer_id"])
    d=category_population.copy(); d["purchased"]=d.customer_id.isin(buyers)
    tab=pd.crosstab(d.gender, d.purchased).reindex(index=["Female","Male"], columns=[False,True], fill_value=0)
    chi2,p,dof,expected=stats.chi2_contingency(tab)
    n=tab.to_numpy().sum(); v=np.sqrt(chi2/(n*min(tab.shape[0]-1,tab.shape[1]-1)))
    rates=d.groupby("gender").purchased.mean()*100
    category_rows.append({"category":cat,"Female %":rates.get("Female",np.nan),"Male %":rates.get("Male",np.nan),"chi2":chi2,"p":p,"Cramers V":v})
gender_results=pd.DataFrame(category_rows)
gender_results["Bonferroni p"]=(gender_results.p*len(gender_results)).clip(upper=1)
gender_results=gender_results.sort_values("Female %")
fig,ax=plt.subplots(figsize=(10,6))
y=np.arange(len(gender_results))
for j,(_,r) in enumerate(gender_results.iterrows()):
    ax.plot([r["Female %"],r["Male %"]],[j,j],color="#9ca3af",linewidth=2)
ax.scatter(gender_results["Female %"],y,label="Female",s=65,color="#286090")
ax.scatter(gender_results["Male %"],y,label="Male",s=65,color="#b45309")
ax.set_yticks(y,gender_results.category); ax.set_xlabel("Matched customers purchasing category at least once (%)")
ax.set_title("Category participation by recorded gender"); ax.legend(); plt.tight_layout(); plt.show()
display(gender_results[["category","Female %","Male %","p","Bonferroni p","Cramers V"]].round(3))


**Interpretation:** In the Week 7 results, Home & Garden had the largest gap (30.16% female vs 23.42% male); the adjusted test remained statistically detectable, but Cramér’s V was very small. The other four category tests were not significant after correction. Statistical detectability does not mean the difference is practically large or actionable.

**FinMark implication:** Treat participation differences as a prompt for further product and customer research. Do not infer preference, cause, or a broad gender-targeting strategy from these small associations.

## 9. Social-media sentiment composition over time

**Question:** How does the composition of recorded sentiment vary by month? The display aggregates `Very Positive` with `Positive`, and `Very Negative` with `Negative`, to avoid overreading the two rare extreme categories. Month-level proportions support comparisons despite varying event counts.


In [ ]:
sentiment_map={"Very Positive":"Positive","Positive":"Positive","Neutral":"Neutral","Negative":"Negative","Very Negative":"Negative"}
social_plot=social.assign(month=social.interaction_date.dt.to_period("M").dt.to_timestamp(), sentiment_group=social.sentiment.map(sentiment_map))
sentiment_pct=(pd.crosstab(social_plot.month,social_plot.sentiment_group,normalize="index")*100).reindex(columns=["Negative","Neutral","Positive"],fill_value=0)
ax=sentiment_pct.plot(kind="bar",stacked=True,figsize=(13,6),color=["#b45309","#9ca3af","#286090"],width=.85)
ax.set(title="Monthly composition of recorded social-media sentiment",xlabel="Interaction month",ylabel="Share of interactions (%)",ylim=(0,100))
ax.legend(title="Grouped sentiment",bbox_to_anchor=(1.02,1),loc="upper left"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("Monthly proportions sum to 100%:", bool(np.allclose(sentiment_pct.sum(axis=1),100)))
display(sentiment_pct.round(1))


**Interpretation:** Week 7 found the three main sentiment labels were roughly balanced overall, while the extreme labels each represented about 1% of records. Grouping preserves the positive/neutral/negative direction while reducing instability from rare labels. Read month-to-month changes as descriptive composition shifts; the one-year social window cannot establish a durable trend or explain why sentiment changed.

**FinMark implication:** Use this chart to identify periods for qualitative follow-up or campaign review, not as a direct measure of customer satisfaction without validation of sentiment labeling.

## 10. Social engagement and purchasing behavior

**Question:** Within the common observation window, do customers with more recorded social interactions also show greater transaction activity or value? Population: customers with both types of records from July 2023 through June 2024. Pearson correlations summarize linear association; one scatterplot shows raw customer-level variation.


In [ ]:
relationship_rows=[]
for label,col in [("Total spending","total_spend"),("Transaction count","transaction_count"),("Average transaction amount","average_transaction_amount")]:
    r,p=stats.pearsonr(matched.interaction_count,matched[col])
    relationship_rows.append({"measure":label,"r":r,"p":p})
relationship=pd.DataFrame(relationship_rows)
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(relationship.r,relationship.measure,s=80,color="#286090")
ax.axvline(0,color="black",linewidth=1); ax.set_xlim(-1,1); ax.set_xlabel("Pearson correlation with interaction count")
ax.set_title("Aligned-period social engagement and purchasing measures"); plt.tight_layout(); plt.show()
display(relationship.round(4))
fig,ax=plt.subplots(figsize=(8,5))
sns.regplot(data=matched,x="interaction_count",y="total_spend",lowess=True,scatter_kws={"alpha":.35,"s":25},line_kws={"color":"#b45309"},ax=ax)
ax.set(title="Customer interactions and total spend (shared period)",xlabel="Social interactions per matched customer",ylabel="Total recorded spend")
plt.tight_layout(); plt.show()


**Interpretation:** On the aligned sample of 564 customers, Week 7 reported correlations of 0.068 for total spending (p=0.1075), 0.055 for transaction count (p=0.1949), and 0.035 for average amount (p=0.4011). These are weak positive linear associations; none is statistically significant at the conventional 0.05 level in those exploratory tests. The data does not support using interaction count as a proxy for customer value. Correlation does not establish that social activity causes purchases.

**FinMark implication:** Do not prioritize customers by raw interaction counts alone. If cross-channel measurement is important, define a stronger engagement measure and evaluate it with campaign timing, exposure, and additional customer context.

## 11. Statistical and analytical validation summary


In [ ]:
validation = pd.DataFrame([
 ["Monthly transaction activity","Monthly count and sum","Transaction-month",f"{monthly_tx.transaction_count.min()}–{monthly_tx.transaction_count.max()} transactions/month","Fluctuation; no seasonality claim"],
 ["Product performance","Category counts and sums","Transaction/category","See category summary above","Compare activity and value; not profit"],
 ["Transaction distribution","Mean, median, SD, IQR rule","Transaction",f"median={amount.median():.2f}; IQR flags={int(((amount<q1-1.5*iqr)|(amount>q3+1.5*iqr)).sum())}","Broad spread; review zeros with context"],
 ["Gender × participation","Chi-square + Bonferroni + Cramér’s V","Matched customer","See category test table","Effect size matters alongside p-value"],
 ["Sentiment over time","Monthly proportions","Interaction-month","Every month sums to 100%","Describe composition, not cause"],
 ["Engagement × purchasing","Pearson r","564 matched customers; aligned period","See correlation table","Weak associations; no causal inference"],
],columns=["Question","Method","Unit/population","Evidence","Interpretation"] )
display(validation)


## 12. Key findings and evidence-based action plan

1. **Transaction activity varies month to month.** Use monthly count and value as monitoring baselines; investigate large deviations against business records. The available period does not validate seasonality.
2. **Category frequency is comparatively balanced.** Compare value and counts together; use additional margin/cost data before making profitability decisions.
3. **Transaction values vary widely.** Report the distribution and median alongside average; preserve the meaning of free/zero transactions.
4. **Gender participation gaps are small.** Continue exploratory analysis with suitable context; avoid using recorded gender as a targeting rule from these results.
5. **Sentiment composition changes across months.** Select periods for review, while validating text labels and collecting more history before making trend claims.
6. **Engagement-purchasing correlations are weak in the matched aligned sample.** Do not use interaction volume alone as a customer-value proxy.

| Evidence | Possible FinMark action | Caution | Success measure for follow-up |
|---|---|---|---|
| Monthly activity fluctuates | Review unusually high/low months with campaign and operations records | Association with events is not yet tested | Monthly count and value with documented context |
| Category counts and value may rank differently | Review both measures in category planning | Revenue is not margin | Category contribution using verified margin data |
| Sentiment shares vary monthly | Review source posts for selected months | Sentiment labels and short history limit inference | Validated sentiment composition over a longer period |
| Weak aligned engagement correlations | Avoid ranking value by interaction count alone | Matched population excludes unmatched customers | Evaluate against defined campaign outcomes |

## 13. Limitations

- Demographics, transactions, and social records have incomplete customer overlap. The aligned cross-channel result uses customers observed in both event sources in the shared period, not all customers. Week 7 reports 564 matched customers for this analysis.
- Transactions span July 2022–June 2024; social interactions span July 2023–June 2024. The comparison is restricted to the common year.
- Customer-level aggregation hides event-level differences and timing within each customer.
- Pearson correlation describes only linear association and does not establish causation.
- Gender tests involve multiple comparisons; Bonferroni correction is applied. Small effects may have limited practical relevance.
- Sentiment extremes are rare and are grouped for monthly visualization.
- Zero amounts include values normalized from explicit `Free` records.
- Signup dates conflict with some transaction dates in earlier review, so tenure analysis is excluded.
- No external product margin, campaign exposure, or customer preference data is available. Conclusions should not be generalized beyond these supplied FinMark datasets.

## 14. Future machine-learning considerations

This milestone does not train a model. Before model selection, FinMark needs a defined prediction target and decision use case. IDs should remain keys, high-cardinality location requires a justified treatment, categorical variables need encoding, and any model must define its eligible customer population. Future work should preserve zero-amount semantics, assess rare sentiment labels if used as a target, and create temporal validation that avoids information leakage.

## 15. Conclusion

The focused visual set extends Milestone 1 and Week 7: it distinguishes transaction frequency from value, shows the wide transaction distribution, evaluates customer participation with an effect-size check, tracks monthly sentiment composition, and tests cross-channel behavior on an aligned matched population. The evidence supports descriptive monitoring and targeted follow-up, not causal claims or strong demographic/engagement targeting. The strongest constraints are incomplete dataset overlap, a short social observation window, and the absence of margin, exposure, and defined prediction-target data.

## 16. Suggested team contribution plan (proposal outline item 21)

The work packages below preserve the proposal’s required section ownership and minimum contribution standard. Names are assigned from the Team 3 roster; the whole team participates in integration and final consistency review.

| Work package | Sections | Expected deliverable | Assigned member |
|---|---|---|---|
| A. Market Activity | 8–9 | Monthly transaction visuals and product-category comparison, with evidence and interpretation | James Lynelle Ongo |
| B. Transaction Behavior | 10 | Transaction distribution analysis, zero/outlier discussion, and interpretation | Kaselyn Cates Ponteres |
| C. Customer Participation | 11 | Gender-category participation, tests, adjusted significance, effect sizes, and chart | Mark Anthony Goyon |
| D. Social and Cross-Channel Behavior | 12–13 | Sentiment trend and aligned engagement-purchasing analysis | Ghaylord Benedict Lugod |
| E. Integration and Quality Review | 6–7, 14–20 | Verification, preparation, synthesis, limitations, conclusion, and final review | Ghaylord Benedict Lugod, with all members reviewing |

Each assigned section should document: analytical question; dataset and unit; preparation; chart rationale; completed chart; numerical evidence; interpretation; business implication; limitation; and successful execution in the final notebook. Confirm individual allocation and contribution records with the team before submission.

## 17. Final validation checklist

- [ ] Notebook executed top to bottom in a fresh Colab runtime.
- [ ] Cleaned datasets and expected columns verified.
- [ ] All charts agree with the tables and written interpretations.
- [ ] Cross-dataset population and common period are visible.
- [ ] Statistical significance and effect size are discussed separately.
- [ ] No causal claims or unsupported seasonality claims remain.
- [ ] Mentor feedback is incorporated once the draft/comments are available.
- [ ] Team contribution entries reflect actual contributions.
